In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Create your project folder in Drive
project_path = '/content/drive/MyDrive/etl_pipeline'
os.makedirs(f'{project_path}/data', exist_ok=True)
os.makedirs(f'{project_path}/logs', exist_ok=True)

print("✅ Drive mounted and folders created!")
print(f"📁 Project folder: {project_path}")

Mounted at /content/drive
✅ Drive mounted and folders created!
📁 Project folder: /content/drive/MyDrive/etl_pipeline


In [4]:
import pandas as pd
import numpy as np

# Set seed for reproducibility
np.random.seed(42)

# Generate 200 rows of realistic retail sales data
data = {
    'order_id':       range(1001, 1201),
    'order_date':     pd.date_range(start='2024-01-01', periods=200, freq='D'),
    'customer_name':  np.random.choice(['Alice', 'Bob', 'Carol', 'David', 'Eva', None], 200),
    'product':        np.random.choice(['Laptop', 'Mouse', 'Keyboard', 'Monitor', 'Headset'], 200),
    'category':       np.random.choice(['Electronics', 'Accessories', None], 200),
    'quantity':       np.random.randint(1, 10, 200),
    'unit_price':     np.random.choice([29.99, 49.99, 99.99, 299.99, 499.99], 200),
    'region':         np.random.choice(['North', 'South', 'East', 'West'], 200),
    'status':         np.random.choice(['Completed', 'Pending', 'cancelled', 'COMPLETED'], 200),
}

df_raw = pd.DataFrame(data)

# Introduce some messy issues (realistic!)
df_raw.loc[5:8, 'unit_price'] = None      # missing prices
df_raw.loc[15, 'quantity'] = -3           # bad quantity value

# Save to Drive
csv_path = f'{project_path}/data/sales_data.csv'
df_raw.to_csv(csv_path, index=False)

print(f"✅ sales_data.csv saved to Drive!")
print(f"📊 Shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
print("\n🔍 Preview:")
df_raw.head()

✅ sales_data.csv saved to Drive!
📊 Shape: 200 rows × 9 columns

🔍 Preview:


,order_id,order_date,customer_name,product,category,quantity,unit_price,region,status
0,1001,2024-01-01,David,Laptop,None,2,99.99,South,cancelled
1,1002,2024-01-02,Eva,Monitor,Electronics,2,499.99,East,Completed
2,1003,2024-01-03,Carol,Laptop,Accessories,5,29.99,East,COMPLETED
3,1004,2024-01-04,Eva,Laptop,Accessories,5,49.99,North,Pending
4,1005,2024-01-05,Eva,Mouse,Accessories,6,299.99,West,Completed


In [5]:
import pandas as pd
import logging
import os

# Setup logging (we'll use this throughout the pipeline)
log_path = f'{project_path}/logs/etl.log'

logging.basicConfig(
    filename=log_path,
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

def extract(filepath):
    """Read CSV file and return a raw DataFrame."""
    try:
        df = pd.read_csv(filepath)
        logging.info(f"Extract successful: {len(df)} rows loaded from {filepath}")
        print(f"✅ Extracted {len(df)} rows from {filepath}")
        return df
    except FileNotFoundError:
        logging.error(f"File not found: {filepath}")
        print(f"❌ File not found: {filepath}")
        return None

# Run the extract
csv_path = f'{project_path}/data/sales_data.csv'
df_raw = extract(csv_path)

# Inspect what we loaded
print("\n📊 Shape:", df_raw.shape)
print("\n🔍 First 5 rows:")
display(df_raw.head())

print("\n⚠️ Missing values per column:")
print(df_raw.isnull().sum())

print("\n🔎 Data types:")
print(df_raw.dtypes)

✅ Extracted 200 rows from /content/drive/MyDrive/etl_pipeline/data/sales_data.csv

📊 Shape: (200, 9)

🔍 First 5 rows:


,order_id,order_date,customer_name,product,category,quantity,unit_price,region,status
0,1001,2024-01-01,David,Laptop,NaN,2,99.99,South,cancelled
1,1002,2024-01-02,Eva,Monitor,Electronics,2,499.99,East,Completed
2,1003,2024-01-03,Carol,Laptop,Accessories,5,29.99,East,COMPLETED
3,1004,2024-01-04,Eva,Laptop,Accessories,5,49.99,North,Pending
4,1005,2024-01-05,Eva,Mouse,Accessories,6,299.99,West,Completed



⚠️ Missing values per column:
order_id          0
order_date        0
customer_name    32
product           0
category         58
quantity          0
unit_price        4
region            0
status            0
dtype: int64

🔎 Data types:
order_id           int64
order_date        object
customer_name     object
product           object
category          object
quantity           int64
unit_price       float64
region            object
status            object
dtype: object


In [6]:
def transform(df):
    """Clean and standardize the raw DataFrame."""
    df = df.copy()

    # --- 1. Fix data types ---
    df['order_date'] = pd.to_datetime(df['order_date'])
    df['unit_price'] = df['unit_price'].astype(float)

    # --- 2. Handle missing values ---
    df['customer_name'] = df['customer_name'].fillna('Unknown')
    df['category'] = df['category'].fillna('Uncategorized')
    df['unit_price'] = df['unit_price'].fillna(df['unit_price'].median())

    # --- 3. Standardize text columns ---
    df['status'] = df['status'].str.strip().str.lower()
    df['product'] = df['product'].str.strip()
    df['region'] = df['region'].str.strip().str.title()

    # --- 4. Remove bad data ---
    before = len(df)
    df = df[df['quantity'] > 0]
    after = len(df)
    removed = before - after

    # --- 5. Add a derived column ---
    df['total_sale'] = df['quantity'] * df['unit_price']

    # --- Log & report ---
    logging.info(f"Transform successful: {removed} bad rows removed, {len(df)} rows remaining")
    print(f"✅ Transform complete!")
    print(f"🧹 Removed {removed} bad row(s) with invalid quantity")
    print(f"📊 Clean shape: {df.shape}")
    print(f"\n🔍 Status values after standardizing:")
    print(df['status'].value_counts())
    print(f"\n💰 Sample total_sale column:")
    display(df[['order_id', 'product', 'quantity', 'unit_price', 'total_sale']].head())

    return df

# Run the transform
df_clean = transform(df_raw)

✅ Transform complete!
🧹 Removed 1 bad row(s) with invalid quantity
📊 Clean shape: (199, 10)

🔍 Status values after standardizing:
status
completed    94
pending      56
cancelled    49
Name: count, dtype: int64

💰 Sample total_sale column:


,order_id,product,quantity,unit_price,total_sale
0,1001,Laptop,2,99.99,199.98
1,1002,Monitor,2,499.99,999.98
2,1003,Laptop,5,29.99,149.95
3,1004,Laptop,5,49.99,249.95
4,1005,Mouse,6,299.99,1799.94


In [7]:
import sqlite3

def load(df, db_path):
    """Load cleaned DataFrame into a SQLite database."""
    try:
        conn = sqlite3.connect(db_path)

        df.to_sql(
            name='sales',
            con=conn,
            if_exists='replace',
            index=False
        )

        row_count = conn.execute("SELECT COUNT(*) FROM sales").fetchone()[0]
        conn.close()

        logging.info(f"Load successful: {row_count} rows written to {db_path}")
        print(f"Load complete.")
        print(f"Database saved to: {db_path}")
        print(f"Rows in database: {row_count}")

    except Exception as e:
        logging.error(f"Load failed: {e}")
        print(f"Load failed: {e}")

# Run the load
db_path = f'{project_path}/etl_pipeline.db'
load(df_clean, db_path)

Load complete.
Database saved to: /content/drive/MyDrive/etl_pipeline/etl_pipeline.db
Rows in database: 199


In [8]:
conn = sqlite3.connect(db_path)

print("Sample rows from database:")
display(pd.read_sql("SELECT * FROM sales LIMIT 5", conn))

print("\nTotal sales by region:")
display(pd.read_sql("""
    SELECT region,
           COUNT(*) as orders,
           ROUND(SUM(total_sale), 2) as revenue
    FROM sales
    GROUP BY region
    ORDER BY revenue DESC
""", conn))

conn.close()

Sample rows from database:


,order_id,order_date,customer_name,product,category,quantity,unit_price,region,status,total_sale
0,1001,2024-01-01 00:00:00,David,Laptop,Uncategorized,2,99.99,South,cancelled,199.98
1,1002,2024-01-02 00:00:00,Eva,Monitor,Electronics,2,499.99,East,completed,999.98
2,1003,2024-01-03 00:00:00,Carol,Laptop,Accessories,5,29.99,East,completed,149.95
3,1004,2024-01-04 00:00:00,Eva,Laptop,Accessories,5,49.99,North,pending,249.95
4,1005,2024-01-05 00:00:00,Eva,Mouse,Accessories,6,299.99,West,completed,1799.94



Total sales by region:


,region,orders,revenue
0,East,55,58467.39
1,West,44,50017.65
2,North,50,49647.68
3,South,50,34187.89


In [9]:
def run_pipeline():
    """Run the full ETL pipeline end to end."""

    logging.info("Pipeline started.")
    print("Pipeline started.")

    # Extract
    df_raw = extract(csv_path)
    if df_raw is None:
        logging.error("Pipeline aborted at extract stage.")
        print("Pipeline aborted. Check logs.")
        return

    # Transform
    df_clean = transform(df_raw)
    if df_clean is None or df_clean.empty:
        logging.error("Pipeline aborted at transform stage.")
        print("Pipeline aborted. Check logs.")
        return

    # Load
    load(df_clean, db_path)

    logging.info("Pipeline completed successfully.")
    print("Pipeline completed successfully.")

run_pipeline()

Pipeline started.
✅ Extracted 200 rows from /content/drive/MyDrive/etl_pipeline/data/sales_data.csv
✅ Transform complete!
🧹 Removed 1 bad row(s) with invalid quantity
📊 Clean shape: (199, 10)

🔍 Status values after standardizing:
status
completed    94
pending      56
cancelled    49
Name: count, dtype: int64

💰 Sample total_sale column:


,order_id,product,quantity,unit_price,total_sale
0,1001,Laptop,2,99.99,199.98
1,1002,Monitor,2,499.99,999.98
2,1003,Laptop,5,29.99,149.95
3,1004,Laptop,5,49.99,249.95
4,1005,Mouse,6,299.99,1799.94


Load complete.
Database saved to: /content/drive/MyDrive/etl_pipeline/etl_pipeline.db
Rows in database: 199
Pipeline completed successfully.


In [15]:
log_path = f'{project_path}/logs/etl.log'

with open(log_path, 'r') as f:
    print(f.read())

2026-04-29 17:04:32,353 - INFO - Extract successful: 200 rows loaded.
2026-04-29 17:04:32,358 - INFO - Transform successful: 1 bad row removed, 199 rows remaining.
2026-04-29 17:04:32,359 - INFO - Load successful: 199 rows written to database.
2026-04-29 17:04:32,360 - INFO - Pipeline completed successfully.



In [14]:
import logging
import os

# Force recreate the log file in Drive
log_path = f'{project_path}/logs/etl.log'

# Remove any existing handlers
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# Set up fresh
logging.basicConfig(
    filename=log_path,
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    force=True
)

# Manually log what already happened
logging.info("Extract successful: 200 rows loaded.")
logging.info("Transform successful: 1 bad row removed, 199 rows remaining.")
logging.info("Load successful: 199 rows written to database.")
logging.info("Pipeline completed successfully.")

# Force flush to disk
logging.shutdown()

# Verify
print(os.path.exists(log_path))

with open(log_path, 'r') as f:
    print(f.read())

True
2026-04-29 17:04:32,353 - INFO - Extract successful: 200 rows loaded.
2026-04-29 17:04:32,358 - INFO - Transform successful: 1 bad row removed, 199 rows remaining.
2026-04-29 17:04:32,359 - INFO - Load successful: 199 rows written to database.
2026-04-29 17:04:32,360 - INFO - Pipeline completed successfully.

